In [0]:
%md
# Migração do ingestion_log para um log por origem

Execução única. Copia as linhas do `fiap_analytics.bronze.ingestion_log` antigo para as tabelas novas, todas no mesmo formato:

1. `fiap_analytics.bronze.cemaden_ingestion_log`
2. `fiap_analytics.bronze.sibh_ingestion_log`
3. `fiap_analytics.bronze.geosampa_ingestion_log`

Regras:

1. Funciona com os dois formatos que o log antigo pode ter (colunas em português ou em inglês).
2. Linhas sem origem identificada (gravadas pelos notebooks em inglês num log em português, que só guardava `batch_id` e `status`) não têm como ser migradas e aparecem só na contagem.
3. Meses das leituras do CEMADEN e das medições do SIBH baixados antes de fechar (mais 3 dias de carência) e gravados como `SUCESSO` ou `SEM_DADOS` passam a `PARCIAL`, para o incremental ou o backfill completá-los.
4. Pode rodar mais de uma vez: linhas já copiadas não são duplicadas.
5. O `ingestion_log` antigo não é apagado. Depois de conferir as tabelas novas, ele pode ser removido com `DROP TABLE fiap_analytics.bronze.ingestion_log`.

Rode antes de qualquer outro notebook de carga.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark.conf.set("spark.sql.session.timeZone", "UTC")  # janelas e horários do log estão em UTC

ANTIGA = "fiap_analytics.bronze.ingestion_log"
DESTINOS = {
    "cemaden": "fiap_analytics.bronze.cemaden_ingestion_log",
    "sibh": "fiap_analytics.bronze.sibh_ingestion_log",
    "geosampa": "fiap_analytics.bronze.geosampa_ingestion_log",
}
CARENCIA_DIAS = 3  # mesma carência dos notebooks de landing

# Mesmo formato do bloco padrão de log dos notebooks de carga.
LOG_SCHEMA = T.StructType([
    T.StructField("batch_id", T.StringType()),
    T.StructField("fonte", T.StringType()),
    T.StructField("etapa", T.StringType()),
    T.StructField("janela", T.StringType()),
    T.StructField("status", T.StringType()),
    T.StructField("registros", T.LongType()),
    T.StructField("paginas", T.IntegerType()),
    T.StructField("duplicados", T.LongType()),
    T.StructField("arquivo", T.StringType()),
    T.StructField("sha256", T.StringType()),
    T.StructField("duracao_s", T.DoubleType()),
    T.StructField("iniciado_em", T.TimestampType()),
    T.StructField("finalizado_em", T.TimestampType()),
    T.StructField("mensagem", T.StringType()),
])
LOG_COLUMNS = [f.name for f in LOG_SCHEMA]


def criar_log(tabela: str) -> None:
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {tabela} (
      batch_id      STRING    COMMENT 'Identificador da execução',
      fonte         STRING    COMMENT 'Conjunto de dados',
      etapa         STRING    COMMENT 'landing ou bronze',
      janela        STRING    COMMENT 'Período processado: mês, data da coleta ou ano do arquivo',
      status        STRING    COMMENT 'SUCESSO, PARCIAL, SEM_DADOS ou FALHA',
      registros     BIGINT,
      paginas       INT       COMMENT 'Chamadas à API',
      duplicados    BIGINT    COMMENT 'Duplicidades pela chave natural do conjunto',
      arquivo       STRING,
      sha256        STRING,
      duracao_s     DOUBLE,
      iniciado_em   TIMESTAMP,
      finalizado_em TIMESTAMP,
      mensagem      STRING
    ) COMMENT 'Controle e checkpoint das cargas do projeto ChuvaSegura, uma tabela por origem'
    """)
    formato = [(f.name, f.dataType) for f in spark.table(tabela).schema]
    if formato != [(f.name, f.dataType) for f in LOG_SCHEMA]:
        raise RuntimeError(f"{tabela} está fora do formato padrão: {formato}")


for tabela in DESTINOS.values():
    criar_log(tabela)

if not spark.catalog.tableExists(ANTIGA):
    dbutils.notebook.exit(f"{ANTIGA} não existe: nada a migrar. As tabelas novas foram criadas.")

In [0]:
antiga = spark.table(ANTIGA)
colunas = set(antiga.columns)

if "fonte" in colunas:  # formato em português (o das leituras do CEMADEN)
    sem_origem = antiga.filter(F.col("fonte").isNull()).count()
    normalizada = antiga.filter(F.col("fonte").isNotNull()).select(
        *[F.col(f.name).cast(f.dataType).alias(f.name) for f in LOG_SCHEMA]
    )
elif "dataset" in colunas:  # formato em inglês (cadastros do CEMADEN e notebooks do SIBH)
    sem_origem = antiga.filter(F.col("dataset").isNull()).count()
    normalizada = antiga.filter(F.col("dataset").isNotNull()).select(
        F.col("batch_id").cast("string").alias("batch_id"),
        F.col("dataset").alias("fonte"),
        F.col("step").alias("etapa"),
        F.col("window").alias("janela"),
        F.when(F.lower("status") == "success", "SUCESSO")
         .when(F.lower("status") == "failed", "FALHA")
         .otherwise(F.upper("status")).alias("status"),
        F.col("records").cast("bigint").alias("registros"),
        F.col("api_calls").cast("int").alias("paginas"),
        F.lit(None).cast("bigint").alias("duplicados"),
        F.col("file_path").alias("arquivo"),
        F.lit(None).cast("string").alias("sha256"),
        F.expr("(unix_micros(finished_at) - unix_micros(started_at)) / 1e6").alias("duracao_s"),
        F.col("started_at").cast("timestamp").alias("iniciado_em"),
        F.col("finished_at").cast("timestamp").alias("finalizado_em"),
        F.col("message").alias("mensagem"),
    )
else:
    raise RuntimeError(f"Formato desconhecido em {ANTIGA}: {sorted(colunas)}")

# Meses baixados antes de fechar (mais a carência) passam a PARCIAL.
inicio_mes = (
    F.when(F.col("fonte") == "cemaden_rainfall_readings", F.expr("try_to_timestamp(janela, 'yyyyMM')"))
     .when(F.col("fonte") == "sibh_measurements", F.expr("try_to_timestamp(janela, 'yyyy-MM')"))
)
mes_completo_em = F.add_months(inicio_mes, 1).cast("timestamp") + F.expr(f"INTERVAL {CARENCIA_DIAS} DAYS")
foi_parcial = (
    (F.col("etapa") == "landing")
    & F.col("status").isin("SUCESSO", "SEM_DADOS")
    & (F.col("iniciado_em") < mes_completo_em)
)
normalizada = normalizada.withColumn(
    "status", F.when(F.coalesce(foi_parcial, F.lit(False)), F.lit("PARCIAL")).otherwise(F.col("status"))
).cache()

print(f"Linhas sem origem identificada (não migradas): {sem_origem}")
display(normalizada.groupBy("fonte", "etapa", "status").count().orderBy("fonte", "etapa", "status"))

In [0]:
CHAVE = ["batch_id", "fonte", "etapa", "janela", "iniciado_em"]
migradas = 0

for prefixo, tabela in DESTINOS.items():
    origem = normalizada.filter(F.col("fonte").startswith(f"{prefixo}_")).alias("o")
    existentes = spark.table(tabela).alias("e")
    novas = origem.join(
        existentes,
        [F.col(f"o.{c}").eqNullSafe(F.col(f"e.{c}")) for c in CHAVE],
        "left_anti",
    ).select(*[F.col(f"o.{c}") for c in LOG_COLUMNS])
    n = novas.count()
    if n:
        novas.write.mode("append").saveAsTable(tabela)
    migradas += n
    print(f"{tabela}: {n} linhas novas")

prefixos = tuple(f"{p}_" for p in DESTINOS)
fora = [r.fonte for r in normalizada.select("fonte").distinct().collect() if not r.fonte.startswith(prefixos)]
if fora:
    print(f"Aviso: origens sem tabela de destino, não migradas: {fora}")

normalizada.unpersist()
print(f"Total migrado: {migradas} linhas")

In [0]:
%sql
SELECT 'cemaden' AS origem, fonte, etapa, status, COUNT(*) AS linhas FROM fiap_analytics.bronze.cemaden_ingestion_log GROUP BY ALL
UNION ALL
SELECT 'sibh', fonte, etapa, status, COUNT(*) FROM fiap_analytics.bronze.sibh_ingestion_log GROUP BY ALL
UNION ALL
SELECT 'geosampa', fonte, etapa, status, COUNT(*) FROM fiap_analytics.bronze.geosampa_ingestion_log GROUP BY ALL
ORDER BY origem, fonte, etapa, status